Importing dataset with 80-20 Class split [Not Fraud ~ 80%, Fraud ~ 20%]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

universe_data = pd.read_csv('/content/Generated_Insurance_Data_Universe.csv')
universe_data

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Intent of Generation,Probability Labels (Fixed Threshold),Probability Labels (Variable Threshold)
0,Gastroenteritis,Female,33,235733.882375,3 days,1,1,1
1,Cataract Surgery,Female,95,82949.783532,1 days,0,0,0
2,Pneumonia,Male,28,70890.840101,4 days,0,0,0
3,Hypertension,Female,98,199713.835387,0 days,1,1,1
4,Migraine,Male,40,30209.892451,2 days,0,0,0
...,...,...,...,...,...,...,...,...
19995,Cancer Treatment,Male,80,780578.369627,6 days,0,0,0
19996,Pregnancy,Female,36,39335.200227,3 days,0,0,0
19997,Pneumonia,Female,15,563636.432404,5 days,1,1,1
19998,Cesarean Section,Female,12,645420.536340,12 days,1,1,1


In [ ]:
universe_data.drop(['Intent of Generation', 'Probability Labels (Variable Threshold)'], axis=1, inplace=True)
universe_data.rename({'Probability Labels (Fixed Threshold)':'Fraud'}, axis=1, inplace=True)
universe_data

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Fraud
0,Gastroenteritis,Female,33,235733.882375,3 days,1
1,Cataract Surgery,Female,95,82949.783532,1 days,0
2,Pneumonia,Male,28,70890.840101,4 days,0
3,Hypertension,Female,98,199713.835387,0 days,1
4,Migraine,Male,40,30209.892451,2 days,0
...,...,...,...,...,...,...
19995,Cancer Treatment,Male,80,780578.369627,6 days,0
19996,Pregnancy,Female,36,39335.200227,3 days,0
19997,Pneumonia,Female,15,563636.432404,5 days,1
19998,Cesarean Section,Female,12,645420.536340,12 days,1


In [ ]:
data = pd.read_csv('/content/Generated_Insurance_Data.csv')
data.drop('Unnamed: 0', axis=1, inplace=True)
data

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Fraud
0,Hypertension,Male,58,59617.156551,2 days,0
1,Cesarean Section,Female,36,690724.893675,3 days,1
2,Pregnancy,Female,39,35777.921812,0 days,0
3,Migraine,Female,27,29326.232971,1 days,0
4,Pregnancy,Female,35,55716.240810,3 days,0
...,...,...,...,...,...,...
1559,Cesarean Section,Female,26,196421.258376,3 days,0
1560,Hypertension,Female,17,51565.918277,0 days,0
1561,Pneumonia,Female,43,161689.687397,3 days,0
1562,Routine Check-up,Female,76,17909.010318,1 days,0


In [ ]:
universe_data['Days in Hospital'] = universe_data['Days in Hospital'].apply(lambda x: int(x.split()[0])).astype(int)
data['Days in Hospital'] = data['Days in Hospital'].apply(lambda x: int(x.split()[0])).astype(int)

In [ ]:
no_fraud = data['Fraud'].value_counts()[0]/len(data)
print("Clean Percentage : ", no_fraud*100)
print("Fraud Percentage : ", (1-no_fraud)*100)

Clean Percentage :  80.1150895140665
Fraud Percentage :  19.884910485933503


In [ ]:
print("Number of Actual Frauds")
print(data['Fraud'].value_counts()[1])

Number of Actual Frauds
311


In [ ]:
from sklearn.metrics import balanced_accuracy_score

Rule Based Classification & Choosing Appropriate Threshold

Rules under soncideration
1) High bill: amt > amt_p75
2) Round bill: amt % 10k ≈ 0
3) Young + long stay: age < 30 & days*age > age_days_cut
4) Elderly + short stay: age > 75 & days <= 2
5) Rare diag + high bill: diag freq < 0.10 & amt > 1.2*amt_p75
6) Female‑twist: female & amt > 1.3*amt_p75

In [ ]:
diag_risk_map = {
    'Cataract Surgery': 0.15,       # routine surgical, low risk
    'Migraine': 0.2,                # moderate recurring
    'Gastroenteritis': 0.25,        # moderately acute
    'Pregnancy': 0.3,               # moderate
    'Routine Check-up': 0.1,        # very low
    'Diabetes': 0.35,               # chronic, moderate-high
    'Cesarean Section': 0.4,        # surgical, higher risk
    'Pneumonia': 0.5,               # acute serious infection
    'Cancer Treatment': 0.6,        # severe condition
    'Hypertension': 0.3             # chronic, moderate
}
gender_twist = {
    ('Female', 'Pregnancy'): 1.3,                  # pregnancy → female boost

    ('Female', 'Migraine'): 1.1,                   # migraines often more severe in women
    ('Male', 'Hypertension'): 0.85,                # slightly lower risk in males

    ('Female', 'Cancer Treatment'): 1.15,          # slightly higher risk for women in some cancers
    ('Male', 'Pneumonia'): 1.1,                    # men worse outcomes

    ('Female', 'Cesarean Section'): 1.25,          # cesarean → obviously female → boost

    ('Male', 'Cataract Surgery'): 0.95,            # slight difference
}

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def get_thresholds(df, target):
    """
    Derive key cut‑offs from the fraud subset to drive our rules:
      - amt_p75: 75th percentile of Amount Billed among frauds
      - diag_freq: frequency of each diagnosis among frauds
      - short_long_cut: median of age*days among frauds
    """
    frauds = df[df[target]==1]
    amt_p75 = frauds['Amount Billed'].quantile(0.75)
    diag_freq = frauds['Diagnosis'].value_counts(normalize=True).to_dict()
    age_days = (frauds['Age'] * frauds['Days in Hospital'])
    short_long_cut = age_days.median()
    return {
        'amt_p75': amt_p75,
        'diag_freq': diag_freq,
        'age_days_cut': short_long_cut
    }

def bank_style_rules(row, th, thresh):
    """
    Apply 6 intuitive flags:
      1) High bill: amt > amt_p75
      2) Round bill: amt % 10k ≈ 0
      3) Young + long stay: age < 30 & days*age > age_days_cut
      4) Elderly + short stay: age > 75 & days <= 2
      5) Rare diag + high bill: diag freq < 0.10 & amt > 1.2*amt_p75
      6) Female‑twist: female & amt > 1.3*amt_p75
    Label fraud if ≥ thresh of these rules fire.
    """
    cnt = 0
    age   = row['Age']
    days  = row['Days in Hospital']
    amt   = row['Amount Billed']
    diag  = row['Diagnosis']
    # 1) High bill
    if amt > th['amt_p75']:
        cnt += 1
    # 2) Round bill
    if abs(amt/10000 - round(amt/10000)) < 1e-6 and amt > th['amt_p75']:
        cnt += 1
    # 3) Young + disproportionately long stay
    if age < 30 and (age * days) > th['age_days_cut']:
        cnt += 1
    # 4) Elderly + very short stay
    if age > 75 and days <= 2:
        cnt += 1
    # 5) Rare diagnosis + premium surcharge
    freq = th['diag_freq'].get(diag, 0)
    if freq < 0.10 and amt > 1.2 * th['amt_p75']:
        cnt += 1
    # 6) Female‑twist on high bills
    if row['Gender']=='Female' and amt > 1.3 * th['amt_p75']:
        cnt += 1
    return 1 if cnt >= thresh else 0

# age_days_series = df['Age'].mul(df['Days in Hospital'].str.split().str[0].astype(int))
# overall_median_age_days = age_days_series.median()
# thresholds = get_thresholds(df)
# df['RatioRuleFraud'] = df.apply(lambda r: ratio_based_rules(r, thresholds['diag_freq']), axis=1)


def apply_bank_rules(df,target,start=2, end=4):
    """
    Add rule‑based predictions for thresholds in [start, end].
    Compute and print accuracy, precision, recall, F1.
    """
    th = get_thresholds(df, target)
    results = {}
    for t in range(start, end+1):
        col = f'RuleFraud_th={t}'
        df[col] = df.apply(lambda r: bank_style_rules(r, th, t), axis=1)
        y_true = df[target]
        y_pred = df[col]
        results[t] = {
            'accuracy' : balanced_accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred),
            'recall'   : recall_score(y_true, y_pred),
            'f1'       : f1_score(y_true, y_pred),
            'cm'       : confusion_matrix(y_true, y_pred),
            'Detected Frauds' : len(df)-df[col].value_counts()[0]
        }
    # display
    for t, m in results.items():
        print(f"Thresh {t:>1d} → "
              f"Acc: {m['accuracy']:.3f}, "
              f"P: {m['precision']:.3f}, "
              f"R: {m['recall']:.3f}, "
              f"F1: {m['f1']:.3f}, "
              f"CM: {m['cm']}")
        print(f"Number of detected frauds {m['Detected Frauds']}")
    return results
    #return df.apply(lambda r: bank_style_rules(r, th, 1), axis=1)

In [ ]:
data

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Fraud
0,Hypertension,Male,58,59617.156551,2,0
1,Cesarean Section,Female,36,690724.893675,3,1
2,Pregnancy,Female,39,35777.921812,0,0
3,Migraine,Female,27,29326.232971,1,0
4,Pregnancy,Female,35,55716.240810,3,0
...,...,...,...,...,...,...
1559,Cesarean Section,Female,26,196421.258376,3,0
1560,Hypertension,Female,17,51565.918277,0,0
1561,Pneumonia,Female,43,161689.687397,3,0
1562,Routine Check-up,Female,76,17909.010318,1,0


In [ ]:
df = data.copy()
df

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Fraud
0,Hypertension,Male,58,59617.156551,2,0
1,Cesarean Section,Female,36,690724.893675,3,1
2,Pregnancy,Female,39,35777.921812,0,0
3,Migraine,Female,27,29326.232971,1,0
4,Pregnancy,Female,35,55716.240810,3,0
...,...,...,...,...,...,...
1559,Cesarean Section,Female,26,196421.258376,3,0
1560,Hypertension,Female,17,51565.918277,0,0
1561,Pneumonia,Female,43,161689.687397,3,0
1562,Routine Check-up,Female,76,17909.010318,1,0


In [ ]:
apply_bank_rules(df, 'Fraud',start=1, end=4)

Thresh 1 → Acc: 0.604, P: 0.374, R: 0.357, F1: 0.365, CM: [[1067  186]
 [ 200  111]]
Number of detected frauds 297
Thresh 2 → Acc: 0.536, P: 0.800, R: 0.077, F1: 0.141, CM: [[1247    6]
 [ 287   24]]
Number of detected frauds 30
Thresh 3 → Acc: 0.500, P: 0.000, R: 0.000, F1: 0.000, CM: [[1252    1]
 [ 311    0]]
Number of detected frauds 1
Thresh 4 → Acc: 0.500, P: 0.000, R: 0.000, F1: 0.000, CM: [[1253    0]
 [ 311    0]]
Number of detected frauds 0


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{1: {'accuracy': np.float64(0.6042347241219145),
  'precision': 0.37373737373737376,
  'recall': 0.35691318327974275,
  'f1': 0.3651315789473684,
  'cm': array([[1067,  186],
         [ 200,  111]]),
  'Detected Frauds': np.int64(297)},
 2: {'accuracy': np.float64(0.5361909552123136),
  'precision': 0.8,
  'recall': 0.07717041800643087,
  'f1': 0.14076246334310852,
  'cm': array([[1247,    6],
         [ 287,   24]]),
  'Detected Frauds': np.int64(30)},
 3: {'accuracy': np.float64(0.49960095770151636),
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'cm': array([[1252,    1],
         [ 311,    0]]),
  'Detected Frauds': np.int64(1)},
 4: {'accuracy': np.float64(0.5),
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'cm': array([[1253,    0],
         [ 311,    0]]),
  'Detected Frauds': np.int64(0)}}

In [ ]:
def apply_bank_rules_decided_thresh(df,target,start=2, end=4):
    """
    Add rule‑based predictions for thresholds in [start, end].
    Compute and print accuracy, precision, recall, F1.
    """
    th = get_thresholds(df, target)
    results = {}
    col = 'RuleFraud_th=1'
    df[col] = df.apply(lambda r: bank_style_rules(r, th, 1), axis=1)
    y_true = df[target]
    y_pred = df[col]
    print('accuracy: ', balanced_accuracy_score(y_true, y_pred))
    print('confusion matrix: ',  confusion_matrix(y_true, y_pred))
    print('Detected Frauds :', len(df)-df[col].value_counts()[0])
    return df

In [ ]:
df = data.copy()

In [ ]:
df = apply_bank_rules_decided_thresh(df, 'Fraud')

accuracy:  0.6042347241219145
confusion matrix:  [[1067  186]
 [ 200  111]]
Detected Frauds : 297


Separating Detected Frauds into a separate dataset to train model

In [ ]:
detected_fraud = df[df['RuleFraud_th=1']==1]
universe = df[df['RuleFraud_th=1']==0]
print(len(detected_fraud), len(universe))

297 1267


Training and testing on the detected fraud data

In [ ]:
from sklearn.linear_model import LogisticRegression

def train_logistic_regression():

    model = LogisticRegression(max_iter=1000)
    return model


In [ ]:
from sklearn.tree import DecisionTreeClassifier

def train_decision_tree():
    model = DecisionTreeClassifier()
    return model


In [ ]:
import lightgbm as lgb

def train_lightgbm():
    model = lgb.LGBMClassifier()

    return model


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
detected_fraud.columns

Index(['Diagnosis', 'Gender', 'Age', 'Amount Billed', 'Days in Hospital',
       'Fraud', 'RuleFraud_th=1'],
      dtype='object')

In [ ]:
X = detected_fraud.drop(['Fraud', 'RuleFraud_th=1'], axis=1)
y = detected_fraud['Fraud']

In [ ]:
import numbers

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
def get_numeric(df):
    num_df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            pass
        else:
            le = LabelEncoder()
            num_df[col] = le.fit_transform(df[col].astype(str))
    return num_df

In [ ]:
X = get_numeric(X)
X

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital
1,2,0,36,6.907249e+05,3
6,8,0,61,7.113161e+05,0
13,3,1,78,4.065721e+04,2
16,0,1,31,9.047045e+05,4
17,9,0,91,1.228711e+04,1
...,...,...,...,...,...
1545,3,0,78,4.259410e+04,2
1551,7,1,8,6.395279e+05,20
1554,0,0,8,7.265205e+05,15
1558,0,1,2,4.139276e+06,20


Trying out Logistic Regression, Decision Tree and LightGBM models on data

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [ ]:
universe_data

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital,Fraud
0,Gastroenteritis,Female,33,235733.882375,3,1
1,Cataract Surgery,Female,95,82949.783532,1,0
2,Pneumonia,Male,28,70890.840101,4,0
3,Hypertension,Female,98,199713.835387,0,1
4,Migraine,Male,40,30209.892451,2,0
...,...,...,...,...,...,...
19995,Cancer Treatment,Male,80,780578.369627,6,0
19996,Pregnancy,Female,36,39335.200227,3,0
19997,Pneumonia,Female,15,563636.432404,5,1
19998,Cesarean Section,Female,12,645420.536340,12,1


In [ ]:
universe_features = universe_data.drop(['Fraud'], axis=1)
universe_target = universe_data['Fraud']
universe_features = get_numeric(universe_features)

In [ ]:
log_reg_model = train_logistic_regression()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
cv_scores = cross_val_score(log_reg_model, X, y, cv=cv, scoring='balanced_accuracy')
log_reg_model.fit(X, y)
y_pred_nfraud = log_reg_model.predict(universe_features)
print(f"Accuracy on Rule-based Fraud: {cv_scores.mean()}")
accuracy_universe = balanced_accuracy_score(universe_target, y_pred_nfraud)
print(f"Accuracy on Universe (Not detected as fraud by rule): {accuracy_universe}")

Accuracy on Rule-based Fraud: 0.8495454264332984
Accuracy on Universe (Not detected as fraud by rule): 0.7447966552554851


In [ ]:
dec_tree_model = train_decision_tree()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=3)
cv_scores = cross_val_score(dec_tree_model, X, y, cv=cv, scoring='balanced_accuracy')
dec_tree_model.fit(X, y)
y_pred_nfraud = dec_tree_model.predict(universe_features)
print(f"Accuracy on Rule-based Fraud: {cv_scores.mean()}")
accuracy_universe = balanced_accuracy_score(universe_target, y_pred_nfraud)
print(f"Accuracy on Universe: {accuracy_universe}")

Accuracy on Rule-based Fraud: 0.9395844461061854
Accuracy on Universe: 0.8210947413297554


In [ ]:
universe_features

,Diagnosis,Gender,Age,Amount Billed,Days in Hospital
0,4,0,33,235733.882375,3
1,1,0,95,82949.783532,1
2,7,1,28,70890.840101,4
3,5,0,98,199713.835387,0
4,6,1,40,30209.892451,2
...,...,...,...,...,...
19995,0,1,80,780578.369627,6
19996,8,0,36,39335.200227,3
19997,7,0,15,563636.432404,5
19998,2,0,12,645420.536340,12


In [ ]:
lgbm_model = train_lightgbm()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lgbm_model, X, y, cv=cv, scoring='balanced_accuracy')
lgbm_model.fit(X, y)
y_pred_nfraud = lgbm_model.predict(universe_features)
print(f"Accuracy on Rule-based Fraud: {cv_scores.mean()}")
accuracy_universe = balanced_accuracy_score(universe_target, y_pred_nfraud)
print(f"Accuracy on Universe: {accuracy_universe}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 88, number of negative: 149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000168 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 150
[LightGBM] [Info] Number of data points in the train set: 237, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.371308 -> initscore=-0.526609
[LightGBM] [Info] Start training from score -0.526609
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

Accuracy obtained on training Decision Tree on full data

In [ ]:
df_features = universe_data.drop(['Fraud'], axis=1)
df_target = universe_data['Fraud']
df_features = get_numeric(df_features)

In [ ]:
dec_tree_model = train_decision_tree()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(dec_tree_model,df_features, df_target, cv=cv, scoring='balanced_accuracy')
print("Accuracy on training on Full Data: ", cv_scores.mean())

Accuracy on training on Full Data:  0.9733972589130119


In [ ]:
lgbm_model = train_lightgbm()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lgbm_model,df_features, df_target, cv=cv, scoring='balanced_accuracy')
print("Accuracy on training on Full Data: ", cv_scores.mean())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3140, number of negative: 12860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000873 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 385
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.196250 -> initscore=-1.409899
[LightGBM] [Info] Start training from score -1.409899
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3141, number of negative: 12859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000590 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_w

In [ ]:
from collections import Counter

In [ ]:
import math
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

target_col   = "Fraud"
batch_size   = 100


X_buffer = X
y_buffer = y

# Stratified k-fold preserves fraud/non-fraud ratio
dec_tree_model = train_decision_tree()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
cv_scores = cross_val_score(dec_tree_model, X, y, cv=cv, scoring='balanced_accuracy')

dec_tree_model.fit(X_buffer, y_buffer)
pred_y = dec_tree_model.predict(universe_features)
acc = balanced_accuracy_score(universe_target, pred_y)
print(f"Initial Cross Validation Accuracy: {cv_scores.mean():.4f}")
print(f"Initial Future Accuracy: {acc:.4f}")


for start in range(0, len(universe)-1, batch_size):
    end   = min(start + batch_size, len(universe)-1)
    batch_feat = universe_features.iloc[start:end]
    batch_targ = universe_target.iloc[start:end]
    rem_x = universe_features.iloc[end:]
    rem_y = universe_target.iloc[end:]

    X_buffer = np.vstack((X_buffer, batch_feat))
    y_buffer = np.hstack((y_buffer, batch_targ))
    dec_tree_model = train_decision_tree()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
    cv_scores = cross_val_score(dec_tree_model, X_buffer, y_buffer, cv=cv, scoring='balanced_accuracy')
    dec_tree_model = train_decision_tree()
    dec_tree_model.fit(X_buffer, y_buffer)
    pred_y = dec_tree_model.predict(rem_x)
    acc = balanced_accuracy_score(rem_y, pred_y)
    print(Counter(y_buffer)[1]/len(y_buffer))

    print(
        f"Additional samples {start}–{end-1} | "
        f"Remaining samples {len(universe)-end} | "
        f"cross_val_acc={cv_scores.mean():.4f}|"
        f"Future Accuracy: {acc:.4f}"
    )


Initial Cross Validation Accuracy: 0.9256
Initial Future Accuracy: 0.8505
0.3224181360201511
Additional samples 0–99 | Remaining samples 1167 | cross_val_acc=0.9070|Future Accuracy: 0.8755
0.2937625754527163
Additional samples 100–199 | Remaining samples 1067 | cross_val_acc=0.9170|Future Accuracy: 0.9033
0.2814070351758794
Additional samples 200–299 | Remaining samples 967 | cross_val_acc=0.9094|Future Accuracy: 0.9050
0.2697274031563845
Additional samples 300–399 | Remaining samples 867 | cross_val_acc=0.9263|Future Accuracy: 0.9194
0.2584692597239649
Additional samples 400–499 | Remaining samples 767 | cross_val_acc=0.9033|Future Accuracy: 0.9108
0.2564102564102564
Additional samples 500–599 | Remaining samples 667 | cross_val_acc=0.9226|Future Accuracy: 0.9074
0.25175526579739216
Additional samples 600–699 | Remaining samples 567 | cross_val_acc=0.9301|Future Accuracy: 0.9190
0.2479489516864175
Additional samples 700–799 | Remaining samples 467 | cross_val_acc=0.9150|Future Accurac